# Important!

- None of the code in the other notebook is technically correct.
- I tried to adapt it from the code given here, but I don't think it works.
- I'm also not sure if it was correct originally. I think the runtime graphs shared with me were in the order of a few milliseconds, while the [Marabou paper](https://arxiv.org/pdf/2401.14461) reports benchmarks on its (presumable complex neural nets) in the order of seconds. 
- I've added explanations for my changed code here, as well as what is incorrect in the old code in the 'VerificationAttempt.ipynb'


### Imports

In [1]:
import numpy as np
from maraboupy import Marabou
import time
import itertools
import onnx
from onnx import compose

Instructions for updating:
non-resource variables are not supported in the long term


### Load Model

In [2]:
layers = '4'
width = '100'

In [3]:
model_path = f'../../models/{layers}_layers/model_{layers}x{width}.onnx'
combined_model_path = f'../../models/{layers}_layers/model_{layers}x{width}_combined.onnx'

- Get neural net feature indices and identify corresponding variables

In [5]:
features = np.load("../../data/features.npy", allow_pickle=True)

print(features[:10], "... and so on.")
print()
print("size:", len(features))

['age' 'fnlwgt' 'education.num' 'capital.gain' 'capital.loss'
 'hours.per.week' 'workclass_ ?' 'workclass_ Federal-gov'
 'workclass_ Local-gov' 'workclass_ Never-worked'] ... and so on.

size: 108


In [6]:
non_categorical = set(range(6))

workclass = set(range(6,15))

education = set(range(15,31))

marital_status = set(range(31,38))

occupation = set(range(38,53))

relationship = set(range(53,59))

race = set(range(59,64))

sex = set(range(64,66))

native_country = set(range(66,108))

all_groups = [workclass, education, marital_status, occupation, relationship, race, sex, native_country]

### Fairness

- Before proceeding, it's important to understand why we need a fairness check for a model like this.
- Suppose a bank uses this model to decide who to grant loans to. If they make more than or equal to $50K, they are more likely to qualify for the loan, and vice versa.
- Now, what if the model learns that people belonging to a certain minority group are more likely to earn less than $50K? It'll give this information to the bank, which then is less likely to sponsor loans to these minority groups, perputuating the cycle. This is unfair.
- We thus want our model to be fair - regardless of changing certain identitarian criteria.

----
- Define functions

In [7]:
# [AI DISCLOSURE - learning and implementing model combination]

def format_model_for_marabou():
    '''
    Combine two identical copies of the model into one graph for Marabou to impose constraints correctly.
    '''

    model_f = onnx.load(model_path)
    
    model_cf = onnx.load(model_path)
    model_cf = compose.add_prefix(model_cf, prefix="CF_")

    combined = compose.merge_models(model_f, model_cf, io_map=[])

    onnx.save(combined, combined_model_path)

format_model_for_marabou()

In [ ]:
### [AI DISCLOSURE: Understanding original code and some modifications necessary]

def setup_network(target_group:set[int], class_A:int, class_B:int):
    '''
    Applies constraints to network for two possible valuations of a categorical feature.
    '''
    net = Marabou.read_onnx(combined_model_path)

    inputs1 = net.inputVars[0][0].tolist()
    inputs2 = net.inputVars[1][0].tolist()

    # ensure no variables have infinite bounds in main or counterfactual network
    # this is also important to ensure the property of CONSISTENCY (lecture 12, slide 37).
    # we do not want our model to create counterexample inputs that can never occur in reality.
    for i in range(len(inputs1)):
        if i in non_categorical:
            lower, upper = -3, 3
            # non categorical features were normalized using StandarScalar, and thus follow
            # the normal distrubution. 99.7% of the data lies within the std dev 3, so we set these bounds
        else:
            lower, upper = 0, 1
            # one hot encoded - variables are either true or false.
        
        net.setLowerBound(inputs1[i], lower)
        net.setUpperBound(inputs1[i], upper)

        net.setLowerBound(inputs2[i], lower)
        net.setUpperBound(inputs2[i], upper)


    # for both networks, add contraints so that multiclass one hot encoding is maintained for examples. 
    for group in all_groups:
        if group == target_group: # skip if target group. needs special handling.
            continue

        group_vars1 = [inputs1[i] for i in group]
        group_vars2 = [inputs2[i] for i in group]
        coeffs = [1] * len(group)
        
        # Add constraint: sum(group_vars) = 1
        net.addEquality(group_vars1, coeffs, 1) # For main net
        net.addEquality(group_vars2, coeffs, 1) # For counterfactual net
    
    # for all inputs except those in the range of the one-hot-encoded classes, 
    # enforce equality between networks inputs1[i] == inputs2[i]
    for i in range(len(inputs1)):

        if i not in target_group:
            net.addEquality([inputs1[i], inputs2[i]], [1, -1], 0)
    
    # CRITICAL FIX: Ensure exactly one race is selected in each network
    # For net1: class_A OR class_B is active (but not both)
    net.addEquality([inputs1[class_A], inputs1[class_B]], [1, 1], 1)
    # A1 != B1
    
    # For net2: class_A OR class_B is active (but not both)  
    net.addEquality([inputs2[class_A], inputs2[class_B]], [1, 1], 1)
    # A2 != B2

    # CRITICAL FIX: Swap the races between the two networks
    net.addEquality([inputs1[class_A], inputs2[class_B]], [1, -1], 0)
    # A1 = B2 => 
    # A1 != A2, B1 != B2
    
    # make sure this applies to only (eg.) white and black persons in both networks.
    for i in (target_group - {class_A, class_B}):

        net.setUpperBound(inputs1[i], 0) 
        net.setUpperBound(inputs2[i], 0)
    
    return net

In [9]:
### [AI DISCLOSURE: Understanding original code and some modifications necessary]

def check_case(case, target_group:set[int], class_A:int, class_B:int, threshold:float, epsilon:float):
    """
    Check one of the fairness violation cases.
    case == 1: net1 output >= threshold and net2 output < threshold
    case == 2: net1 output < threshold and net2 output >= threshold
    """
    start_time = time.time()
    

    #for class_A, class_B in list(itertools.combinations(target_group, 2)):
    # Setup networks and constraints common to both
    net = setup_network(target_group, class_A, class_B)
    
    # Retrieve the output variables
    output1 = int(net.outputVars[0][0])
    output2 = int(net.outputVars[1][0])

    
    if case == 1:
        net.setLowerBound(output1, threshold)
        net.setUpperBound(output2, threshold - epsilon)
    elif case == 2:
        net.setUpperBound(output1, threshold - epsilon)
        net.setLowerBound(output2, threshold)
    else:
        raise ValueError("Case must be 1 or 2.")
    
    exitCode, vals, stats = net.solve()
    elapsed_time = time.time() - start_time
    
    return exitCode, vals, elapsed_time

In [ ]:
# Adapted original code

def fairness_check(target_group:set[int], threshold:float, epsilon:float):
    '''
    Check all pairs in a given categorical class to see if any of them provide a SATisfying counterexample
    '''
    
    for class_A, class_B in list(itertools.combinations(target_group, 2)):
        
        exitCode1, vals1, time1 = check_case(1, target_group, class_A, class_B, threshold, epsilon)
        print("Time Taken (Case 1) =", round(time1, 4), "seconds", flush = True)
        if exitCode1 == "sat":
            print(f"Fairness violation found (Case 1): Found a pair of classes ({class_A}, {class_B}) which change the outcome if swapped.", flush = True)
            return vals1

        else:
            exitCode2, vals2, time2 = check_case(2, target_group, class_A, class_B, threshold, epsilon)
            print("Time Taken (Case 2) =", round(time2, 4), "seconds", flush = True)
            
            if exitCode2 == "sat":
                print(f"Fairness violation found (Case 2): Found a pair of classes ({class_A}, {class_B}) which change the outcome if swapped.", flush = True)
                return vals2

            else:
                pass
    
    print(f"No fairness violations were found for any of the classes in {target_group}", flush = True)
    return None

# Overview

- Marabou cannot fully verify two neural networks at once - which we need for our counterfactual fairness (one set of inputs vs another set inputs, mildly changed, in the other.) See previous notebooks for why.

- To allow for this, we pack the two identical networks into the same onnx graph, where neither are connected to each other.

- We then implement our constraints within this single network, which Marabou should be able to verify.

- Unfortunately, the following set of constraints seems to be too large for marabou to handle:

In [ ]:
'''for i in range(len(inputs1)):
        if i in non_categorical:
            lower, upper = -3, 3
            # non categorical features were normalized using StandarScalar, and thus follow
            # the normal distrubution. 99.7% of the data lies within the std dev 3, so we set these bounds
        else:
            lower, upper = 0, 1
            # one hot encoded - variables are either true or false.
        
        net.setLowerBound(inputs1[i], lower)
        net.setUpperBound(inputs1[i], upper)

        net.setLowerBound(inputs2[i], lower)
        net.setUpperBound(inputs2[i], upper)'''

-  I ran the code with this section present for 6 hours and failed to get an output. With all other contraints removed and only this one kept, it still runs indefinitely. This section is the bottleneck.

- Keeping this part is necessary - without it, Marabou generates an error, saying that there are variables with infinite bounds.

- It's possible that it acts as such a heavy bottleneck because I'm checking a graph containg with two (duplicate) models. But it seems that even on a single model, everything slows to a halt - see **Robustness**.

- So, while I am much more confident theoretically about my implementation, I can't say it's functional.

- I might just be doing everything wrong - Marabou's own documentation is for version 1.0.0 and is quite outdated. I've adapted to the best of my ability.

#### Test Delay with Race

Let $race(x)$ denote the race one-hot subvector of $x$, and
let $RaceVals$ be the set of valid race encodings. For any $x \in \mathcal{X}$ define $x'$ to be $x$ with $race(x)$
replaced (all other coordinates unchanged). The model is fair at $x$ *iff*

$$\forall \ race(x) \in RaceVals \ \left( \^{y}(x) = \^{y}(x') \right)$$

**Verification target**: search for a violation—i.e., SAT of

$$\exists \ x \in \mathcal{X}, \ \exists \ race(x) \in RaceVals \ \left( \^{y}(x) \neq \^{y}(x') \right)$$



In [ ]:
fairness_check(race, threshold = 0.5, epsilon = 1e-3)

# should take ...very long.

### Robustness

For $ y \in \{0, 1\} $, let $ \mu_y $ be the mean input (in normalized feature space) over the subset with label $ y $. Fix $ \epsilon > 0 $ and norm $ \|\cdot\|_\infty $. Define the admissible $ \epsilon $-ball:

$$
B_\infty(\mu_y, \epsilon) = \{x : \|x - \mu_y\|_\infty \leq \epsilon, \text{ continuous features within bounds, categoricals remain valid one-hot}\},
$$

The model is **robust at** $ \mu_y $ **iff**:

$$
\forall x \in B_\infty(\mu_y, \epsilon) : \hat{y}(x) = \hat{y}(\mu_y),
$$

In [ ]:
### Adapted from Original Notebook code.

def epsilon_ball(epsilon = 0.01):

    options = Marabou.createOptions(verbosity = 0)

    net = Marabou.read_onnx(model_path)

    # Load x_avg from file
    x_avg = np.load("../../data/x_avg.npy")

    # Access the input variable list; adjust indexing based on your network structure.
    # Here we assume net.inputVars[0][0] is an array of input variable IDs.
    inputs = net.inputVars[0][0].tolist()

    # Set epsilon as some percentage of the average for each input dimension.
    for i, var in enumerate(inputs):

        epsilon_i = epsilon * abs(x_avg[i])

        net.setLowerBound(var, x_avg[i] - epsilon_i)
        net.setUpperBound(var, x_avg[i] + epsilon_i)
    
    # we do not want our model to create counterexample inputs that can never occur in reality.
    for i in range(len(inputs)):
        if i in non_categorical:
            lower, upper = -3, 3
            # non categorical features were normalized using StandarScalar, and thus follow
            # the normal distrubution. 99.7% of the data lies within the std dev 3, so we set these bounds
        else:
            lower, upper = 0, 1
            # one hot encoded - variables are either true or false.
        
        net.setLowerBound(inputs[i], lower)
        net.setUpperBound(inputs[i], upper)
    
    # for both networks, add contraints so that multiclass one hot encoding is maintained for examples. 
    for group in all_groups:

        group_vars1 = [inputs[i] for i in group]
        coeffs = [1] * len(group)
        
        # Add constraint: sum(group_vars) = 1
        net.addEquality(group_vars1, coeffs, 1) # For net1
        
    # Now we have set our input constraints to be within some %age of x_avg.
    exitCode, vals, stats = net.solve(options = options)

    print ("MODEL PATH:", model_path)
    if exitCode == "unsat":
        print("The network is robust in the ε-ball around the average input.")
    else:
        print("Found a counterexample within the ε-ball that changes the classification.")
        print("Counterexample:", vals)

# Overview

- A similar heavy delay failure occurs in the case of robustness, with the added constraints 
on the One-Hot-Encoded inputs to ensure that they are either 0 or 1.
- These are crucial for consistency. The offending codeblock has been outlined below.

In [ ]:
''' for i in range(len(inputs)):
        if i in non_categorical:
            lower, upper = -3, 3
            # non categorical features were normalized using StandarScalar, and thus follow
            # the normal distrubution. 99.7% of the data lies within the std dev 3, so we set these bounds
        else:
            lower, upper = 0, 1
            # one hot encoded - variables are either true or false.
        
        net.setLowerBound(inputs[i], lower)
        net.setUpperBound(inputs[i], upper)'''

#### Test Delay Hypothesis with Robustness

In [ ]:
epsilon_ball(0.01) # set epsilon = 1%

# should take... very long.

# Guess on Runtime Issues

- My guess is that Marabou is optimized for real-valued, scalar inputs. Our network, however, has one-hot-encoded categorical features. Adding constraints on these becomes a Boolean SAT task, which the tool is likely not optimized for. 

- Examples discussed in the class (aircraft collision avoidance, image classification) as well as benchmark neural nets on the [Marabou 2.0 paper](https://arxiv.org/pdf/2401.14461) all appear to encode non-boolean input vectors as well.